# Imports

In [1]:
import pandas as pd
import numpy as np
import math

from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupShuffleSplit

In [2]:
data_dir = "./data"
seed = 42

In [3]:
model = CatBoostRanker()
model.load_model('model.cbm')

CatBoostRanker(depth=5, eval_metric='NDCG:top=10', iterations=1000, learning_rate=0.05, loss_function='YetiRank', od_type='Iter', od_wait=200, random_seed=42, verbose=100)

In [4]:
model.get_params()

{'eval_metric': 'NDCG:top=10',
 'od_wait': 200,
 'od_type': 'Iter',
 'iterations': 1000,
 'verbose': 100,
 'loss_function': 'YetiRank',
 'depth': 5,
 'random_seed': 42,
 'learning_rate': 0.05}

# Data

In [3]:
train_df = pd.read_csv(f"{data_dir}/train.csv")
test_df = pd.read_csv(f"{data_dir}/test.csv")

In [4]:
train_df.shape

(223044, 69)

In [5]:
train_df.Label.unique()

array([2, 0, 3, 1])

In [6]:
display(train_df.head())
display(test_df.head())

,QueryId,Label,q_len,q_uniq,q_idf_sum,q_idf_max,q_idf_min,q_has_digit,q_has_latin,q_n_candidates,...,bm25_title_minus_mean,bm25_body_rank,bm25_body_to_max,bm25_body_minus_mean,coverage_body_rank,coverage_body_to_max,coverage_body_minus_mean,lm_body_rank,lm_body_to_max,lm_body_minus_mean
0,1,2,4.0,4.0,20.08,5.924,3.767,0.0,0.0,31.0,...,4.590,0.5161,0.75800,0.8122,0.54840,0.75,0.00,0.6452,1.471,-0.06894
1,1,0,4.0,4.0,20.08,5.924,3.767,0.0,0.0,31.0,...,-9.626,1.0000,0.09637,-24.2300,0.98390,0.25,-0.50,1.0000,2.330,-16.19000
2,1,2,4.0,4.0,20.08,5.924,3.767,0.0,0.0,31.0,...,5.193,0.6452,0.74160,0.1938,0.54840,0.75,0.00,0.5161,1.453,0.25350
3,1,0,4.0,4.0,20.08,5.924,3.767,0.0,0.0,31.0,...,1.999,0.1290,0.91920,6.9170,0.08065,1.00,0.25,0.1290,1.280,3.50800
4,1,2,4.0,4.0,20.08,5.924,3.767,0.0,0.0,31.0,...,-5.842,0.7419,0.72310,-0.5058,0.54840,0.75,0.00,0.8065,1.556,-1.66400


,ObjectId,QueryId,q_len,q_uniq,q_idf_sum,q_idf_max,q_idf_min,q_has_digit,q_has_latin,q_n_candidates,...,bm25_title_minus_mean,bm25_body_rank,bm25_body_to_max,bm25_body_minus_mean,coverage_body_rank,coverage_body_to_max,coverage_body_minus_mean,lm_body_rank,lm_body_to_max,lm_body_minus_mean
0,1,2,4.0,4.0,24.14,13.35,0.4823,0.0,0.0,38.0,...,-0.4472,0.42110,0.7614,-0.4166,0.6711,0.6667,-0.07895,0.57890,1.155,-0.8116
1,2,2,4.0,4.0,24.14,13.35,0.4823,0.0,0.0,38.0,...,5.9270,0.02632,1.0000,4.7160,0.1711,1.0000,0.17110,0.05263,1.011,4.3800
2,3,2,4.0,4.0,24.14,13.35,0.4823,0.0,0.0,38.0,...,0.2877,0.68420,0.7437,-0.7966,0.6711,0.6667,-0.07895,0.39470,1.144,-0.4204
3,4,2,4.0,4.0,24.14,13.35,0.4823,0.0,0.0,38.0,...,0.8929,0.71050,0.7429,-0.8145,0.6711,0.6667,-0.07895,0.92110,1.174,-1.5140
4,5,2,4.0,4.0,24.14,13.35,0.4823,0.0,0.0,38.0,...,-0.6292,0.39470,0.7624,-0.3941,0.6711,0.6667,-0.07895,0.42110,1.144,-0.4409


# Utils

In [ ]:
def make_pool(df, features, label, group_id):
    return Pool(
        data=df[features],
        label=df[label],
        group_id=df[group_id]
    )


def dcg(gains, k=10):
    return sum(g / math.log2(i + 2) for i, g in enumerate(gains[:k]))  # позиция i с нуля -> log2(i + 2)


def ndcg(gains, all_gains, k=10):
    ideal = dcg(sorted(all_gains, reverse=True), k)
    return dcg(gains, k) / ideal if ideal > 0 else 0.0


def get_ndcg_score(model, pool, df, k=10):
    preds = model.predict(pool)
    df = df.assign(Prediction=preds)
    grouped = df.groupby('QueryId')
    ndcg_scores = []
    for query_id, group in grouped:
        labels = group['Label'].values
        order = np.argsort(-group['Prediction'].values, kind='stable')
        gains = labels[order]
        ndcg_scores.append(ndcg(gains, labels, k))

    return np.mean(ndcg_scores)

# X and y

In [8]:
features = [column for column in train_df.columns if column not in ('QueryId', 'Label')]

In [9]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(train_df, groups=train_df['QueryId']))
train_idx, val_idx = np.sort(train_idx), np.sort(val_idx)

train_part, val_part = train_df.iloc[train_idx].copy(), train_df.iloc[val_idx].copy()

train_pool = make_pool(train_part, features, 'Label', 'QueryId')
val_pool = make_pool(val_part, features, 'Label', 'QueryId')

# Train Model

In [40]:
params = dict(
    loss_function='LambdaMart',
    eval_metric='NDCG:top=10',
    iterations=1000,
    learning_rate=0.05,
    depth=5,
    random_seed=seed,
    verbose=25
)

In [41]:
model = CatBoostRanker(**params)
model.fit(
    train_pool,
    eval_set=val_pool,
    early_stopping_rounds=200,
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	test: 0.6384743	best: 0.6384743 (0)	total: 26.5ms	remaining: 26.5s
25:	test: 0.7268527	best: 0.7268785 (24)	total: 455ms	remaining: 17s
50:	test: 0.7280360	best: 0.7284169 (48)	total: 939ms	remaining: 17.5s
75:	test: 0.7298650	best: 0.7298650 (75)	total: 1.36s	remaining: 16.6s
100:	test: 0.7309090	best: 0.7310736 (99)	total: 1.85s	remaining: 16.5s
125:	test: 0.7314614	best: 0.7315714 (113)	total: 2.3s	remaining: 15.9s
150:	test: 0.7315203	best: 0.7316536 (147)	total: 2.73s	remaining: 15.3s
175:	test: 0.7325752	best: 0.7325752 (175)	total: 3.16s	remaining: 14.8s
200:	test: 0.7329549	best: 0.7329549 (200)	total: 3.57s	remaining: 14.2s
225:	test: 0.7337158	best: 0.7337533 (223)	total: 3.99s	remaining: 13.7s
250:	test: 0.7340876	best: 0.7343707 (242)	total: 4.4s	remaining: 13.1s
275:	test: 0.7342733	best: 0.7343707 (242)	total: 4.81s	remaining: 12.6s
300:	test: 0.7349268	best: 0.7353374 (287)	total: 5.22s	remaining: 12.1s
325:	test: 0.7352511	best: 0.7353374 (287)	total: 5.64s	remaining

CatBoostRanker(depth=5, eval_metric='NDCG:top=10', iterations=1000, learning_rate=0.05, loss_function='LambdaMart', random_seed=42, verbose=25)

# Eval Model

In [36]:
train_ndcg_score = get_ndcg_score(model, train_pool, train_part, k=10)
val_ndcg_score = get_ndcg_score(model, val_pool, val_part, k=10)

In [37]:
train_ndcg_score, val_ndcg_score

(np.float64(0.7687233075165237), np.float64(0.7391654290441766))